In [23]:
import numpy as np
import matplotlib.pyplot as plt
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [24]:
from keras.datasets import mnist # type: ignore

(xTrain,yTrain), (xTest,yTest) = mnist.load_data() #60k train, 10k test
def flattener(X):
    return np.reshape(X, (len(X), 28*28))
def reshaper(X):
    return np.reshape(X,(len(X), 28,28))

xTrain = flattener(xTrain)
xTest = flattener(xTest)
yTrain = torch.LongTensor(yTrain)
yTest = torch.LongTensor(yTest)

In [25]:
import torch.functional as F
from torch import nn

adversarial_loss = nn.BCELoss()
class Discriminator(nn.Module):
    def __init__(self, dims_in):
        super().__init__()
        self.dims_in = dims_in
        self.model = nn.Sequential(
            nn.Linear(self.dims_in, 128),
            nn.LeakyReLU(0.1),
            nn.Linear(128,1),
            nn.Tanh())
        
    def forward(self,x):
        return self.model(x)
        
class Generator(nn.Module):
    def __init__(self, dims_in, latent_dim):
        super().__init__()
        self.dims_in = dims_in
        self.latent_dim = latent_dim
        self.model = nn.Sequential(
            nn.Linear(self.latent_dim, 256),
            nn.LeakyReLU(0.1),
            nn.Linear(256,self.dims_in),
            nn.Tanh())
          
    def forward(self,x):
        return self.model(x)

In [26]:
import torch.optim.adam
import torchvision
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

lr = 3e-4
latent_dim = 64
image_dim = 28*28
batch_size = 32
n_iter = 1

transforms = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.1307,) , (0.3081,))])
dataset = datasets.MNIST(transform=transforms, download=True, root="datasets/")
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

criterion = nn.BCELoss()
discr = Discriminator(image_dim)
gen = Generator(image_dim, latent_dim)
optim_D = torch.optim.Adam(discr.parameters(), lr=lr)
optim_G = torch.optim.Adam(gen.parameters() , lr=lr)

In [27]:
for i in range(n_iter):

    for (batch_ids, (real_imgs, _)) in enumerate(loader):
    
        real_imgs = torch.reshape(real_imgs, (-1,784)) #-1 makes it automatically adjust the no. of images
        noise = torch.randn(batch_size,latent_dim)
        fake = gen(noise)
        
        real_d = discr(real_imgs)
        real_d = torch.flatten(real_d)
        loss_real_d = criterion(real_d, torch.ones_like(real_d))

        fake_d = discr(fake.detach())
        fake_d = torch.flatten(fake_d)
        loss_fake_d = criterion(fake_d, torch.zeros_like(fake_d))

        lossD = loss_real_d + loss_fake_d
        discr.zero_grad()
        lossD.backward()
        optim_D.step()

        output = torch.flatten(fake_d)
        lossG = criterion(output, torch.ones_like(fake_d))
        gen.zero_grad()
        lossG.backward()
        optim_G.step()

RuntimeError: all elements of input should be between 0 and 1